# 03 · Data Cleaning

Step-by-step cleaning of the raw customer data: junk values, duplicates, inconsistent categories, mixed number and date formats, outliers and missing-value imputation. The result is saved to `data/processed/clean_customers.csv`.

## 1. Imports and loading

In [1]:
import re

import numpy as np
import pandas as pd

RAW_CSV = "../data/raw/raw_customers.csv"
CLEAN_CSV = "../data/processed/clean_customers.csv"

df = pd.read_csv(RAW_CSV)
print(f"Rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

Rows loaded: 550
Columns: ['customer_id', 'age', 'gender', 'country', 'channel', 'monthly_spend', 'clicks', 'impressions', 'conversions', 'signup_date']


## 2. Snapshot of the problem *before* cleaning

Always document the initial state so the impact of the cleaning is measurable.

In [2]:
print("=== INITIAL STATE ===")
print(f"Total rows          : {len(df)}")
print(f"Duplicate rows      : {df.duplicated(subset=['age','gender','country','channel','monthly_spend']).sum()}")
print(f"\nEmpty values (empty string ''):")
print((df == "").sum())
print(f"\n'N/A'-like values:")
print(df.isin(["N/A", "none", "unknown", "??"]).sum())

=== INITIAL STATE ===
Total rows          : 550
Duplicate rows      : 51

Empty values (empty string ''):
customer_id      0
age              0
gender           0
country          0
channel          0
monthly_spend    0
clicks           0
impressions      0
conversions      0
signup_date      0
dtype: int64

'N/A'-like values:
customer_id       0
age              12
gender            0
country           0
channel           0
monthly_spend     0
clicks            0
impressions       0
conversions       0
signup_date       0
dtype: int64


## 3. Step 1 — Standardize empty strings and junk → `NaN`

Anything that is not real data becomes `NaN`, so pandas treats it uniformly (`isnull`, `fillna`, `dropna`).

In [3]:
JUNK = ["N/A", "na", "n/a", "none", "unknown", "??", "", " "]

df.replace(JUNK, np.nan, inplace=True)

print("Real missing values after standardizing:")
print(df.isnull().sum())

Real missing values after standardizing:
customer_id        0
age               63
gender            50
country           24
channel           23
monthly_spend    178
clicks            21
impressions        0
conversions        0
signup_date        0
dtype: int64


## 4. Step 2 — Remove duplicates

Keep the first occurrence and drop the rest.

In [4]:
before = len(df)
df.drop_duplicates(
    subset=["age", "gender", "country", "channel", "monthly_spend"],
    keep="first",
    inplace=True
)
df.reset_index(drop=True, inplace=True)
print(f"Rows before: {before} → after: {len(df)}  (removed: {before - len(df)})")

Rows before: 550 → after: 499  (removed: 51)


## 5. Step 3 — Normalize `gender`

Goal: keep only three values → `Male`, `Female`, `Other`.

In [5]:
def normalize_gender(value):
    if pd.isnull(value):
        return np.nan
    v = str(value).strip().lower()
    if v in ["male", "m", "man"]:
        return "Male"
    elif v in ["female", "f", "woman"]:
        return "Female"
    elif v in ["other", "nb", "non-binary"]:
        return "Other"
    else:
        return np.nan   # unrecognized values → missing

df["gender"] = df["gender"].apply(normalize_gender)
print("Unique values in gender after cleaning:")
print(df["gender"].value_counts(dropna=False))

Unique values in gender after cleaning:
gender
Male      210
Female    194
Other      51
NaN        44
Name: count, dtype: int64


## 6. Step 4 — Normalize `country`

`Mexico`, `mexico`, `MX`, `MEXICO` → `Mexico`

In [6]:
country_map = {
    "mexico": "Mexico", "méxico": "Mexico", "mx": "Mexico",
    "colombia": "Colombia", "col": "Colombia",
    "argentina": "Argentina", "arg": "Argentina",
    "chile": "Chile", "cl": "Chile",
    "peru": "Peru", "perú": "Peru", "pe": "Peru",
}

def normalize_country(value):
    if pd.isnull(value):
        return np.nan
    v = str(value).strip().lower()
    return country_map.get(v, np.nan)   # not in the map → missing

df["country"] = df["country"].apply(normalize_country)
print("Unique values in country after cleaning:")
print(df["country"].value_counts(dropna=False))

Unique values in country after cleaning:
country
Mexico       149
Colombia     109
Argentina     89
Chile         66
Peru          46
NaN           40
Name: count, dtype: int64


## 7. Step 5 — Normalize `channel`

`Instagram`, `instagram`, `INSTAGRAM` → `Instagram`

In [7]:
channel_map = {
    "instagram": "Instagram",
    "facebook": "Facebook", "fb": "Facebook",
    "google": "Google", "google ads": "Google",
    "tiktok": "TikTok", "tik tok": "TikTok",
    "email": "Email", "e-mail": "Email",
}

def normalize_channel(value):
    if pd.isnull(value):
        return np.nan
    v = str(value).strip().lower()
    return channel_map.get(v, np.nan)

df["channel"] = df["channel"].apply(normalize_channel)
print("Unique values in channel after cleaning:")
print(df["channel"].value_counts(dropna=False))

Unique values in channel after cleaning:
channel
Instagram    133
Facebook     128
Google        92
TikTok        67
Email         58
NaN           21
Name: count, dtype: int64


## 8. Step 6 — Clean `monthly_spend`

`"$1,234.56"`, `"1234 USD"`, `"1234.56"` → `1234.56` (float). A regex keeps only the number.

In [8]:
def clean_spend(value):
    if pd.isnull(value):
        return np.nan
    # Remove $, commas, spaces, letters (USD, etc.)
    cleaned = re.sub(r"[^\d.]", "", str(value))
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

df["monthly_spend"] = df["monthly_spend"].apply(clean_spend)
print("monthly_spend after cleaning:")
print(df["monthly_spend"].describe())

monthly_spend after cleaning:
count     337.000000
mean     2540.314095
std      1449.626994
min       106.040000
25%      1264.530000
50%      2572.200000
75%      3866.000000
max      4996.000000
Name: monthly_spend, dtype: float64


## 9. Step 7 — Convert numeric columns

`age`, `clicks`, `impressions` and `conversions` are numbers, not strings. `errors="coerce"` turns anything unparseable into `NaN`.

In [9]:
for col in ["age", "clicks", "impressions", "conversions"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
customer_id       object
age              float64
gender            object
country           object
channel           object
monthly_spend    float64
clicks           float64
impressions        int64
conversions        int64
signup_date       object
dtype: object


## 10. Step 8 — Detect and handle outliers in `clicks`

We use the **IQR (interquartile range)** method: `Q1 − 1.5·IQR` and `Q3 + 1.5·IQR` define the normal range.

In [10]:
Q1 = df["clicks"].quantile(0.25)
Q3 = df["clicks"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["clicks"] < lower_bound) | (df["clicks"] > upper_bound)]
print(f"Outliers in clicks: {len(outliers)} rows")
print(f"Normal range: [{lower_bound:.0f}, {upper_bound:.0f}]")
print(f"Outlier values found:\n{outliers['clicks'].values}")

# Replace outliers with NaN (we do not drop the whole row)
df.loc[df["clicks"] > upper_bound, "clicks"] = np.nan
df.loc[df["clicks"] < lower_bound, "clicks"] = np.nan
print(f"\nOutliers replaced with NaN ✓")

Outliers in clicks: 7 rows
Normal range: [-582, 1554]
Outlier values found:
[106583. 162651. 146049.  62058. 133637.  52370. 199333.]

Outliers replaced with NaN ✓


## 11. Step 9 — Standardize dates

Five different formats → a single ISO format (`YYYY-MM-DD`).

In [11]:
df["signup_date"] = pd.to_datetime(
    df["signup_date"],
    errors="coerce"    # anything unparseable → NaT (missing date)
)
print("Dates after standardizing:")
print(df["signup_date"].describe())
print(f"Unparsed dates (NaT): {df['signup_date'].isnull().sum()}")

Dates after standardizing:
count                               86
mean     2023-12-25 16:27:54.418604544
min                2023-01-06 00:00:00
25%                2023-06-24 00:00:00
50%                2024-01-20 00:00:00
75%                2024-06-18 00:00:00
max                2024-11-22 00:00:00
Name: signup_date, dtype: object
Unparsed dates (NaT): 413


## 12. Step 10 — Impute remaining missing values

Strategy: **median** for numeric columns (robust to outliers), **mode** for categorical columns (most frequent value).

In [12]:
# Numeric → median
for col in ["age", "monthly_spend", "clicks"]:
    median = df[col].median()
    df[col] = df[col].fillna(median)
    print(f"{col}: missing → median ({median:.1f})")

# Categorical → mode
for col in ["gender", "country", "channel"]:
    mode = df[col].mode()[0]
    df[col] = df[col].fillna(mode)
    print(f"{col}: missing → mode ('{mode}')")

age: missing → median (41.0)
monthly_spend: missing → median (2572.2)
clicks: missing → median (476.5)
gender: missing → mode ('Male')
country: missing → mode ('Mexico')
channel: missing → mode ('Instagram')


## 13. Snapshot *after* cleaning

Compare against the initial state.

In [13]:
print("=== FINAL STATE ===")
print(f"Total rows          : {len(df)}")
print(f"Remaining missing   : {df.isnull().sum().sum()}")
print(f"\nUnique values in country : {sorted(df['country'].dropna().unique())}")
print(f"Unique values in gender  : {sorted(df['gender'].dropna().unique())}")
print(f"Unique values in channel : {sorted(df['channel'].dropna().unique())}")
print(f"\nNumeric column statistics:")
print(df[["age","monthly_spend","clicks","impressions","conversions"]].describe().round(1))

=== FINAL STATE ===
Total rows          : 499
Remaining missing   : 413

Unique values in country : ['Argentina', 'Chile', 'Colombia', 'Mexico', 'Peru']
Unique values in gender  : ['Female', 'Male', 'Other']
Unique values in channel : ['Email', 'Facebook', 'Google', 'Instagram', 'TikTok']

Numeric column statistics:
         age  monthly_spend  clicks  impressions  conversions
count  499.0          499.0   499.0        499.0        499.0
mean    41.4         2550.7   482.7      24783.8         25.3
std     12.6         1190.8   287.5      13996.2         14.7
min     18.0          106.0     1.0        525.0          0.0
25%     31.0         1892.8   231.5      13252.0         12.0
50%     41.0         2572.2   476.5      23475.0         25.0
75%     52.0         3222.9   733.5      37038.0         39.0
max     64.0         4996.0   995.0      49971.0         49.0


## 14. Save for the next notebook

In [14]:
df.to_csv(CLEAN_CSV, index=False)
print(f"✓ Saved to '{CLEAN_CSV}'")

✓ Saved to '../data/processed/clean_customers.csv'
